<a href="https://colab.research.google.com/github/ercastil/mapbiomas_testing/blob/main/colab/Copy_of_mapbiomas_fire_mosaics_cog_chile_b24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 👉 Set up the environment
# (Optional) Install dependencies if not already available.

#!apt-get install -y gdal-bin python3-gdal  # Installs GDAL tools
#!pip install gdal termcolor  # Installs GDAL and termcolor Python packages

In [ ]:
# Authenticate with your Google account to access Google Cloud resources
from google.colab import auth
auth.authenticate_user()  # Prompts user login for authentication

# Import required libraries
import os  # For handling file system operations
import subprocess  # For running system commands like 'gsutil'
from osgeo import gdal  # GDAL for geospatial data processing
from termcolor import colored  # For printing colored output in the terminal


In [ ]:
# Parameters
countries = ["chile"] #Options: ["bolivia", "colombia", "chile", "ecuador","peru", "paraguay", "guyana","suriname","venezuela"]
regions = ['2']

# Define which satellites to use for which years
satellite_years = [
    #{'satellite': 'b24_', 'years': [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025]},
    {'satellite': 'b24', 'years': [2024]},
]

# Base paths for processing (used in Colab)
base_dir = '/content/mosaics_temp'

# Main loop through countries, regions, satellites, and years
for country in countries:
    for region in regions:
        for satellite_info in satellite_years:
            satellite = satellite_info['satellite']
            for year in satellite_info['years']:
                print(colored(f"\n🔄 Procesando: Country = {country}, Region = {region}, Year = {year}, Satellite = {satellite}", 'cyan'))

                # Define naming and paths
                prefix = f'{satellite}_{country}_r{region}_{year}'
                input_folder = f'{base_dir}/input'
                output_vrt = f'{base_dir}/{prefix}_mosaic.vrt'
                output_cog_local = f'{base_dir}/output_cog/{prefix}_cog.tif'
                output_cog_bucket = f'gs://mapbiomas-fire/sudamerica/{country}/collection1/24b/mosaics_cog/{prefix}_cog.tif'

                # Create necessary directories
                for folder in [input_folder, os.path.dirname(output_vrt), os.path.dirname(output_cog_local)]:
                    os.makedirs(folder, exist_ok=True)

                # Copy .tif files from bucket to local input folder
                print(colored(f"📥 Copiando archivos con prefijo '{prefix}'...", 'yellow'))
                try:
                    subprocess.run([
                        "gsutil", "-m", "cp",
                        f"gs://mapbiomas-fire/sudamerica/{country}/collection1/24b/mosaics/{prefix}*",
                        input_folder
                    ], check=True)
                except subprocess.CalledProcessError:
                    print(colored("❌ Error al copiar los archivos.", 'red'))
                    continue

                # List downloaded .tif files
                file_list = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith('.tif') and f.startswith(prefix)]

                # Skip if no files found
                if not file_list:
                    print(colored("⚠️ No se encontraron archivos .tif. Saltando...", 'red'))
                    continue
                else:
                    print(colored(f"🔎 {len(file_list)} archivos encontrados para el mosaico.", 'green'))

                # Create VRT (virtual raster)
                try:
                    vrt_options = gdal.BuildVRTOptions(resampleAlg='nearest', addAlpha=False)
                    gdal.BuildVRT(output_vrt, file_list, options=vrt_options)
                    print(colored(f"🔧 VRT creado: {output_vrt}", 'green'))
                except Exception as e:
                    print(colored(f"❌ Error al crear el VRT: {e}", 'red'))
                    continue

                # Convert VRT to Cloud-Optimized GeoTIFF (COG)
                try:
                    cog_options = gdal.TranslateOptions(
                        format="COG",
                        creationOptions=[
                            "COMPRESS=LZW",
                            "PREDICTOR=2",       # Best for continuous data (spectral bands)
                            "ZLEVEL=9",          # Max DEFLATE compression level
                            "RESAMPLING=AVERAGE",
                            "BIGTIFF=YES"
                        ]
                    )
                    gdal.Translate(output_cog_local, output_vrt, options=cog_options)
                except Exception as e:
                    print(colored(f"❌ Error al generar el COG: {e}", 'red'))
                    continue

                # Upload COG to the bucket
                try:
                    subprocess.run(["gsutil", "cp", output_cog_local, output_cog_bucket], check=True)
                    print(colored(f"☁️ COG enviado al bucket: {output_cog_bucket}", 'green'))
                except subprocess.CalledProcessError:
                    print(colored(f"❌ Error al subir al bucket: {output_cog_bucket}", 'red'))
                    continue

                # Clean up temporary files
                try:
                    for f in file_list:
                        os.remove(f)
                    os.remove(output_vrt)
                    os.remove(output_cog_local)
                    print(colored("🧹 Limpieza completada.", 'green'))
                except Exception as e:
                    print(colored(f"⚠️ Error durante la limpieza: {e}", 'red'))
